In [1]:
# !pip install --upgrade langfuse openai opentelemetry-sdk opentelemetry-exporter-otlp

In [2]:
# !pip install langfuse

# 参考文档
- FastMcp: https://gofastmcp.com/clients/client
- Mcp: https://modelcontextprotocol.io/docs

In [3]:
# 环境准备
# pin install fastmcp

# Mcp Client调用chrome-devtools

In [4]:
# stdio的方式创建client
import asyncio
from fastmcp import Client, FastMCP
from fastmcp.client.transports import StdioTransport
import json
import pprint

transport = StdioTransport(
    command="npx",
    args=["chrome-devtools-mcp@latest", "--autoConnect", "--channel=beta"]
)
mcp_client = Client(transport)

- 工具调用测试

In [5]:
# 列出mcp server的工具
async with mcp_client: 
    tools = await mcp_client.list_tools()
    tools = [tool for tool in tools if tool.name!='take_screenshot'] # tool返回截图，以base64编码返回，容易出现token超限
    print(*tools, sep="\n")

name='click' title=None description='Clicks on the provided element' inputSchema={'type': 'object', 'properties': {'uid': {'type': 'string', 'description': 'The uid of an element on the page from the page content snapshot'}, 'dblClick': {'type': 'boolean', 'description': 'Set to true for double clicks. Default is false.'}, 'includeSnapshot': {'type': 'boolean', 'description': 'Whether to include a snapshot in the response. Default is false.'}}, 'required': ['uid'], 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'} outputSchema=None icons=None annotations=ToolAnnotations(title=None, readOnlyHint=False, destructiveHint=None, idempotentHint=None, openWorldHint=None, category='input') meta=None execution=ToolExecution(taskSupport='forbidden')
name='close_page' title=None description='Closes the page by its index. The last open page cannot be closed.' inputSchema={'type': 'object', 'properties': {'pageId': {'type': 'number', 'description': 'The ID of the p

In [6]:
# 执行打开新页面的工具
async with mcp_client: 
    await mcp_client.call_tool("new_page", {"url": "https://cloud.siliconflow.cn/me/models", "background": False, "timeout": 30000})

- 大模型调用

In [7]:
# from openai import OpenAI
from langfuse.openai import OpenAI

from dotenv import load_dotenv
import json 
import os
from langfuse import Langfuse

load_dotenv()

langfuse = Langfuse()

# 生成随机合法 trace_id
trace_id = langfuse.create_trace_id()

ai_client = OpenAI(
    base_url=os.environ['OPENAI_BASE_URL'],
    api_key=os.environ['OPENAI_API_KEY']
)

In [10]:
def converter(tool_obj):
    schema = {
        "type": "function",
        "function": {
            "name": tool_obj.name,
            "description": tool_obj.description,
            "parameters": tool_obj.inputSchema
        }
    }
    return schema

In [11]:
# mcp tool转换成open ai tool schema
async with mcp_client:
    ai_tools = [converter(mcp_tool) for mcp_tool in tools]

In [14]:
# 通过mcp client调用工具
messages = [{
    "role": "user",
    "content": "请帮我查下余额充值中的代金券，https://cloud.siliconflow.cn/me/expensebill"
    }]

finish_reason = None 
round = 0
while finish_reason == None or finish_reason == "tool_calls":
    round += 1
    resp = ai_client.chat.completions.create(
        model="moonshotai/Kimi-K2-Thinking",
        messages=messages,
        tools = ai_tools,
        temperature=1.0,
        tool_choice="auto",
        trace_id=trace_id
    )
    choices = resp.choices[0]
    tool_calls = choices.message.tool_calls
    finish_reason = resp.choices[0].finish_reason
    if finish_reason == "tool_calls":
        print(f"Round {round}: {tool_calls}" )
        messages.append(choices.message)
        for tool_call in tool_calls:
            tool_name = tool_call.function.name 
            print(f"chorme-devtool: {tool_name}")
            tool_arguments = tool_call.function.arguments
            async with mcp_client: 
                tool_call_result = await mcp_client.call_tool(tool_name, json.loads(tool_arguments), timeout=60)
                tool_call_result_content = "\n".join([content.text for content in  tool_call_result.content]) 
            messages.append(
                {
                    'role': 'tool',
                    'tool_call_id': tool_call.id,
                    "name": tool_call.function.name,
                    'content': tool_call_result_content
                }
            )
print(resp.choices[0].message.content)

Round 1: [ChatCompletionMessageFunctionToolCall(id='functions.new_page:0', function=Function(arguments='{"url": "https://cloud.siliconflow.cn/me/expensebill", "timeout": 30000}', name='new_page'), type='function')]
chorme-devtool: new_page
Round 2: [ChatCompletionMessageFunctionToolCall(id='functions.take_snapshot:1', function=Function(arguments='{"verbose": true}', name='take_snapshot'), type='function')]
chorme-devtool: take_snapshot
Round 3: [ChatCompletionMessageFunctionToolCall(id='functions.click:2', function=Function(arguments='{"uid": "1_137", "includeSnapshot": true}', name='click'), type='function')]
chorme-devtool: click
根据查询结果，您的代金券信息如下：

**代金券概况：**
- 总数量：1 张可用
- 代金券名称：模型服务代金券（2025 版）
- 初始金额：¥14.00
- 剩余额度：¥3.7155
- 状态：可用

**详细信息：**
- **可用范围**：相关模型的在线体验、API 请求或微调训练
- **有效期至**：2099-12-31 23:59:59（长期有效）
- **代金券编号**：W202512141312231006000005435801

您当前可以使用这张代金券来抵扣相关模型的在线体验、API 请求或微调训练的费用，剩余可用额度为 ¥3.7155 元。
